# 머신러닝 미니 프로젝트
## 정형데이터 이진 분류
- 심장병 발병 여부 예측
  - 데이터 출처
    - Kaggle (Heart Disease Dataset / 원출처: UCI Machine Learning Repository)
    - https://www.kaggle.com/datasets/arezaei81/heartcsv?resource=download
  

In [ ]:
import numpy as np
import pandas as pd
from tensorflow import keras
from keras.layers import Dense
from keras.models import Sequential
from keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# 데이터 세트를 읽기(결측치 제거)
df = pd.read_csv("/content/heart.csv", na_values='?')

# 입력 변수(X)와 출력 변수(y) 분리
# [:,0:13] -> 전체 행 가져고오고 0번부터 12번의 열만 갖고옴
X = df.iloc[:, 0:13]
# [:.13] -> 전체 행 가져오고 13 열만 선택하여 Y로 출력
y = df.iloc[:, 13]

# 이진분류로 만들기 (0-5를 0과 1로 변환)
# 0 -> 정상, 1 -> 심장병 발병
y = np.where(y > 0, 1, 0)

# 최소-최대 스케일러 객체 생성
# (데이터 모든 값 0-1 사이의 값으로 변환)
mms = MinMaxScaler()
mms.fit(X) # 최소값-최대값 기억
print("정규화 전 X 데이터:\n", X.head())

# 실제 데이터를 0-1사이 값을 실행
X_mms = mms.transform(X)
# 예쁘게 표로 정리
X = pd.DataFrame(X_mms, columns=X.columns, index=list(X.index.values))
print("정규화 후 X 데이터:\n", X.head())

# 훈련 데이터(70%)와 테스트 데이터(30%) 분할 (-> 과적합 방지)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=True, random_state=42)
print("출력 변수(y) 분포:\n", pd.Series(y).value_counts())

정규화 전 X 데이터:
    age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   3       145   233    1        0      150      0      2.3      0   
1   37    1   2       130   250    0        1      187      0      3.5      0   
2   41    0   1       130   204    0        0      172      0      1.4      2   
3   56    1   1       120   236    0        1      178      0      0.8      2   
4   57    0   0       120   354    0        1      163      1      0.6      2   

   ca  thal  
0   0     1  
1   0     2  
2   0     2  
3   0     2  
4   0     2  
정규화 후 X 데이터:
         age  sex        cp  trestbps      chol  fbs  restecg   thalach  exang  \
0  0.708333  1.0  1.000000  0.481132  0.244292  1.0      0.0  0.603053    0.0   
1  0.166667  1.0  0.666667  0.339623  0.283105  0.0      0.5  0.885496    0.0   
2  0.250000  0.0  0.333333  0.339623  0.178082  0.0      0.0  0.770992    0.0   
3  0.562500  1.0  0.333333  0.245283  0.251142  0.0      0.5  0.816794    0.0

In [ ]:
# 모델의 이름을 정합니다.
modelpath='/content/heart_modelr.keras'

# 케라스 모델을 생성한다. (이진 분류 모델)
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=(13,)))  # 입력층: 13개 속성
model.add(keras.layers.Dense(16, activation='relu'))
model.add(keras.layers.Dense(32, activation='relu'))
model.add(keras.layers.Dense(32, activation='relu'))
model.add(keras.layers.Dense(16, activation='relu'))
# 출력층: 이진 분류이므로 뉴런 1개 + sigmoid
model.add(keras.layers.Dense(1, activation='sigmoid'))

# 케라스 모델을 컴파일한다.
model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

# 콜백 설정 (20회 이상 검증 손실이 향상되지 않으면 조기 종료)
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=20)


# 모델 학습 실행 (훈련 데이터의 25%를 검증셋으로 사용)
history = model.fit(X_train, y_train,
                    validation_split=0.25,
                    epochs=2000,
                    batch_size=32,
                    callbacks=[early_stopping_callback, checkpointer])

# 테스트 결과를 출력한다.
score = model.evaluate(X_test, y_test)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

Epoch 1/2000
1/5 ━━━━━━━━━━━━━━━━━━━━ 12s 3s/step - accuracy: 0.5625 - loss: 0.6893
Epoch 1: val_loss did not improve from 0.34539
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 130ms/step - accuracy: 0.6478 - loss: 0.6878 - val_accuracy: 0.6604 - val_loss: 0.6871
Epoch 2/2000
1/5 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.6562 - loss: 0.6815
Epoch 2: val_loss did not improve from 0.34539
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.6289 - loss: 0.6824 - val_accuracy: 0.6226 - val_loss: 0.6817
Epoch 3/2000
1/5 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.6562 - loss: 0.6761
Epoch 3: val_loss did not improve from 0.34539
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.5912 - loss: 0.6776 - val_accuracy: 0.6226 - val_loss: 0.6768
Epoch 4/2000
1/5 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.6250 - loss: 0.6709
Epoch 4: val_loss did not improve from 0.34539
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5723 - loss: 0.6726 - val_accuracy: 0.6604 - val_loss: 0.6712
Epoch 5/2000
1/5